# 12. Agentic RAG: Answering with Tools

**RAG Pipeline Series — Notebook 12**

[Notebook 11](11_end_to_end_pipeline_gradio.ipynb) wired retrieval, augmentation, and generation into one `rag_answer()` function: every question goes through the *same* fixed path — retrieve from `rag.pdf`, stuff the chunks into a prompt, generate. That works well as long as `rag.pdf` actually contains the answer. It breaks down the moment a question falls outside the document — a question about something that happened after the document was written, or a topic the course simply doesn't cover.

This notebook replaces that fixed path with an **agent**: instead of always retrieving first, the model itself decides, per question, which of two *tools* to call:

1. **`search_rag_document`** — the same dense retriever from notebooks 1-10, wrapped as a tool, for questions about the RAG course material.
2. **`web_search`** — a free, key-less web search (via [DDGS](https://pypi.org/project/ddgs/), the DuckDuckGo search library) for anything current or outside the document.

The model can call either tool, both, or neither, and can chain calls (e.g. check the document first, then fall back to the web) before producing a final answer. This is the essence of *agentic RAG*: retrieval becomes one tool among several, invoked at the model's discretion rather than unconditionally.

The user-facing workflow is unchanged from notebook 11 — a question goes in, a grounded answer comes out via the same Gradio chat UI. What changed is what happens in between. The notebook is written to run standalone in Google Colab as well as locally.

## Setup

In [ ]:
%pip install -q -U langchain langchain-community langchain-core langchain-text-splitters sentence-transformers langchain-huggingface langchain-chroma chromadb langchain-google-genai python-dotenv gradio pandas ddgs

### Files this notebook needs

- `rag.pdf` — the source document (same as every other notebook in the series).
- `.env` — must contain a `GOOGLE_API_KEY` for Gemini. Get a free key from [Google AI Studio](https://aistudio.google.com/apikey); the `gemini-2.5-flash` model used below is available on the free tier.

No key is needed for web search — `ddgs` queries DuckDuckGo directly, without an API key.

In Colab neither file exists on the VM yet, so the cell below opens a file picker twice — once for `rag.pdf`, once for `.env` — via the same `maybe_colab_upload()` helper the series already uses. Locally, both files already sit next to this notebook, so the picker is skipped.

In [ ]:
from rag_utils import maybe_colab_upload

# Only runs inside Colab. Opens a file picker each time; select rag.pdf, then .env.
# Safe to skip this cell if you're running locally and already have both files on disk.
maybe_colab_upload()  # -> rag.pdf
maybe_colab_upload()  # -> .env

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv

# Colab uploads land in /content; locally the .env sits next to this notebook.
env_path = "/content/.env" if os.path.exists("/content/.env") else ".env"
assert os.path.exists(env_path), "Could not find .env - upload it first (see cell above)."
load_dotenv(env_path)

assert os.environ.get("GOOGLE_API_KEY"), "GOOGLE_API_KEY not set - check your .env file."
print("GOOGLE_API_KEY loaded.")

GOOGLE_API_KEY loaded.


## 1. Rebuild the retriever

Same pipeline as notebooks 7-11: load `rag.pdf`, strip headers/footers, chunk chapter-by-chapter, embed with the series' `sentence-transformers/paraphrase-MiniLM-L3-v2` model, and index into an in-memory Chroma store.

In [2]:
from rag_utils import build_chroma_store, get_embedder, load_chapter_chunks

pages, full_text, chapters, chunks = load_chapter_chunks()

embeddings = get_embedder()
vectorstore = build_chroma_store(chunks, embeddings=embeddings, collection_name="rag_pdf_chapters")
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

print(f"{len(chunks)} chunks indexed; retriever returns top 5 matches per query")

d:\youtube\TheAIGuy\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 55/55 [00:00<00:00, 5358.20it/s]


181 chunks indexed; retriever returns top 5 matches per query


## 2. Writing tools

A LangChain tool is just a Python function with a `@tool` decorator, a type-hinted signature, and a docstring. The docstring matters more than usual here: it's not documentation for a human, it's the *only* description the model sees when deciding whether to call the tool. A vague docstring produces a model that calls the wrong tool, or none at all.

Two tools:

- **`search_rag_document`** wraps the `retriever` built above. Its docstring tells the model this is the place to look for anything about the RAG course itself — chunking, retrieval, embeddings, re-ranking, augmentation, generation.
- **`web_search`** wraps `ddgs.DDGS().text()`, a free DuckDuckGo text search that needs no API key. Its docstring tells the model this is for anything current or outside the document.

Each tool returns a plain string (the model only understands text), but also logs what it did into `tool_call_log` — a module-level list the Gradio UI reads afterward to show *which* tools fired and what they found, so the agent's reasoning isn't a black box.

In [3]:
from langchain_core.tools import tool

tool_call_log = []  # reset per question in rag_agent_answer(); UI reads this for transparency


def format_context(docs):
    return "\n\n".join(
        f"[Chapter {d.metadata['chapter_num']}: {d.metadata['chapter_title']}]\n{d.page_content}"
        for d in docs
    )


@tool
def search_rag_document(query: str) -> str:
    """Search rag.pdf, the RAG course document, for relevant passages.

    Use this for any question about RAG concepts covered in the course: chunking,
    embeddings, vector stores, keyword/dense/hybrid retrieval, re-ranking,
    augmentation, or generation. Always try this tool first for conceptual
    questions about RAG - it is the authoritative source for the course material.
    """
    docs = retriever.invoke(query)
    result = format_context(docs)
    tool_call_log.append({
        "tool": "search_rag_document",
        "query": query,
        "summary": ", ".join(f"Chapter {d.metadata['chapter_num']}" for d in docs),
    })
    return result


@tool
def web_search(query: str) -> str:
    """Search the live web for current information that rag.pdf would not contain.

    Use this for questions about recent events, current facts, or any topic outside
    the RAG course document - anything time-sensitive or not related to RAG concepts.
    """
    from ddgs import DDGS

    with DDGS() as ddgs:
        results = list(ddgs.text(query, max_results=3))

    tool_call_log.append({
        "tool": "web_search",
        "query": query,
        "summary": ", ".join(r["title"] for r in results) if results else "no results",
    })

    if not results:
        return "No web results found."
    return "\n\n".join(f"{r['title']}\n{r['body']}\nSource: {r['href']}" for r in results)


TOOLS = [search_rag_document, web_search]
TOOLS_BY_NAME = {t.name: t for t in TOOLS}

## 3. Binding tools to Gemini

`llm.bind_tools(TOOLS)` returns a new runnable that, on `invoke()`, may return an `AIMessage` with a `tool_calls` list instead of (or alongside) plain text - Gemini's way of saying "call this function with these arguments before you can answer." The model decides whether to call a tool at all; nothing here forces it to.

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", temperature=0.2)
llm_with_tools = llm.bind_tools(TOOLS)

## 4. The agent loop: `rag_agent_answer()`

This is the agentic replacement for notebook 11's straight-line `rag_answer()`. Instead of a fixed retrieve → augment → generate sequence, it's a loop:

1. Send the conversation so far to the model.
2. If the model responds with `tool_calls`, run each requested tool locally and append its result back into the conversation as a `ToolMessage`, then go to step 1 - giving the model a chance to call another tool (e.g. check the document, come up empty, then search the web) or produce a final answer.
3. If the model responds with plain text and no `tool_calls`, that's the final answer - stop.

`max_steps` is a safety valve: without it, a model that never stops calling tools would loop forever.

In [5]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

AGENT_SYSTEM_PROMPT = (
    "You are a helpful assistant that answers questions using tools.\n"
    "- For questions about RAG concepts (chunking, retrieval, embeddings, re-ranking, "
    "augmentation, generation), call search_rag_document first.\n"
    "- For questions about current events, facts, or anything outside the RAG course "
    "document, call web_search.\n"
    "- If search_rag_document doesn't contain the answer, try web_search before giving up.\n"
    "- Answer only from tool results - do not rely on outside knowledge you were not given "
    "by a tool. Mention which tool(s) and, if applicable, which chapter(s) your answer came from."
)


def rag_agent_answer(question, max_steps=4):
    tool_call_log.clear()
    messages = [SystemMessage(content=AGENT_SYSTEM_PROMPT), HumanMessage(content=question)]

    for _ in range(max_steps):
        ai_message = llm_with_tools.invoke(messages)
        messages.append(ai_message)

        if not ai_message.tool_calls:
            return ai_message.content, list(tool_call_log)

        for tool_call in ai_message.tool_calls:
            selected_tool = TOOLS_BY_NAME[tool_call["name"]]
            result = selected_tool.invoke(tool_call["args"])
            messages.append(ToolMessage(content=result, tool_call_id=tool_call["id"]))

    return "Reached the tool-call limit without a final answer.", list(tool_call_log)

Try it on two questions before wiring up a UI: one squarely inside `rag.pdf` (should trigger `search_rag_document`), one clearly outside it (should trigger `web_search`).

In [6]:
answer, log = rag_agent_answer("What is re-ranking used for in a RAG pipeline?")
print("ANSWER:\n", answer)
print("\nTOOL CALLS:")
for entry in log:
    print(f"  - {entry['tool']}({entry['query']!r}) -> {entry['summary']}")

ANSWER:
 [{'type': 'text', 'text': 'In a RAG pipeline, **re-ranking** is used as a precision-focused second stage in a multi-stage retrieval funnel to identify the most relevant documents from a larger set of initial candidates.\n\nAccording to **Chapter 8 (Re-ranking)** of the RAG course document, its primary functions and characteristics include:\n\n*   **Improving Precision:** While the first stage of retrieval (using methods like BM25 or vector search) is "recall-focused" and quickly gathers a wide net of candidates (e.g., top 100–500), the re-ranking stage is "precision-focused." It uses a more powerful model to surgically select the most relevant documents (typically narrowing them down to the top 5–10).\n*   **Cross-Encoder Models:** Re-ranking typically employs **cross-encoder models**. These are more computationally expensive than the bi-encoders used in initial retrieval but are much more accurate at determining the specific relevance of a document to a query.\n*   **Balancin

In [7]:
answer, log = rag_agent_answer("Who won the last fifa world cup?")
print("ANSWER:\n", answer)
print("\nTOOL CALLS:")
for entry in log:
    print(f"  - {entry['tool']}({entry['query']!r}) -> {entry['summary']}")

ANSWER:
 [{'type': 'text', 'text': 'Argentina won the last FIFA World Cup, which took place in 2022. They defeated France in the final on December 18, 2022, winning 4-2 on penalties after the match ended in a 3-3 draw.\n\nThis information was found using the **web_search** tool.'}]

TOOL CALLS:
  - web_search('who won the last fifa world cup') -> List of FIFA World Cup finals - Wikipedia, 2022 FIFA World Cup - Wikipedia, FIFA World Cup Winners List 1930–2026 Complete History
  - web_search('winner of 2022 FIFA World Cup') -> 2022 FIFA World Cup final - Wikipedia, 2022 FIFA World Cup - Wikipedia, FIFA World Cup champions: 1982-2022
  - web_search('who won 2022 FIFA World Cup final Argentina vs France') -> 2022 FIFA World Cup final - Wikipedia, Argentina 3-3 France (Dec 18, 2022) Final Score - ESPN, Argentina wins incredible World Cup final in a shootout with France - CNBC


## 5. A Gradio UI

Same shape as notebook 11's UI - a question box, an "Ask" button, and an answer box - plus one addition: a "Tool calls" box that surfaces `tool_call_log`, so it's visible which tool(s) the agent chose and why, instead of a fixed "retrieved chapters" list.

`demo.launch(share=True, debug=True)` is the Colab-safe way to launch: Colab can't serve a local port directly, so Gradio needs `share=True` to tunnel the UI to a public URL it prints below the cell. This works locally too - it just also opens the tunnel there.

In [15]:
import gradio as gr


def ask(question):
    if not question or not question.strip():
        return "Please enter a question.", ""
    answer, log = rag_agent_answer(question)
    if log:
        tool_calls = "\n".join(f"- {e['tool']}({e['query']!r}) -> {e['summary']}" for e in log)
    else:
        tool_calls = "(no tools called - answered directly)"
    if isinstance(answer, list) and len(answer) > 0:
        answer = answer[0]['text']
    elif isinstance(answer, str):
        answer = answer
    else:
        answer = "Unexpected answer format."

    return answer, tool_calls


with gr.Blocks(title="Agentic RAG over rag.pdf + the web") as demo:
    gr.Markdown(
        "# Ask rag.pdf (or the web)\n"
        "Agentic RAG: the model chooses between the notebook 1-10 retriever and a free "
        "web search tool, then generates a grounded answer (this notebook)."
    )
    question_box = gr.Textbox(label="Your question", placeholder="e.g. What is re-ranking used for?")
    ask_btn = gr.Button("Ask")
    answer_box = gr.Textbox(label="Answer", lines=6)
    tool_calls_box = gr.Textbox(label="Tool calls", lines=4)

    ask_btn.click(fn=ask, inputs=question_box, outputs=[answer_box, tool_calls_box])
    question_box.submit(fn=ask, inputs=question_box, outputs=[answer_box, tool_calls_box])

demo.launch(share=True, debug=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://6e5cb89e1f6d68a5ad.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://6e5cb89e1f6d68a5ad.gradio.live


## Takeaways

- A **tool** is a plain Python function plus a `@tool` decorator, a type-hinted signature, and a docstring the model reads to decide when to call it - the docstring is a prompt, not documentation.
- `llm.bind_tools([...])` turns any chat model into one that can *request* function calls; the model decides whether and which tool to call, nothing forces it.
- The **agent loop** (invoke → check for `tool_calls` → run tools → feed results back → repeat) is what turns a single retrieve-then-generate pipeline into one that can chain multiple tools, or skip retrieval entirely, per question.
- Retrieval doesn't disappear in agentic RAG - it becomes `search_rag_document`, one tool the model can choose alongside others like `web_search`, rather than a step every question is forced through.
- The workflow the user sees is unchanged from notebook 11 - a question in, an answer out via Gradio - but what happens in between now adapts to the question instead of following one fixed path.

This closes out the series with the pattern most production RAG systems actually use: retrieval as a callable tool inside a broader agent, not a hardcoded first step.